<a href="https://colab.research.google.com/github/Jimpang77/Feature-Data-Trade-Off/blob/main/Research_Data_Training_Result.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://github.com/Jimpang77


https://www.nature.com/articles/s41598-024-51825-x





https://link.springer.com/article/10.1186/s40537-024-00886-w


# Adult Census Income — Data Cleaning

**This is a classification problem.** The goal is to predict whether a person's annual income is **above \$50K** (`>50K` vs `<=50K`) — a **binary classification**, not a regression (we are not predicting a continuous number).

This notebook is the **preprocessing foundation** for the main study (the Feature vs. Data trade-off). We diagnose each dirty part of the data and fix it, one step at a time.

**What is dirty in this data (confirmed):**
- Missing values are stored as the **string `?`**, not as `NaN` (in `workclass`, `occupation`, `native_country`)
- The raw UCI file has **leading spaces** in strings (e.g. `' Private'`)
- There are **duplicate rows**
- `education` and `education_num` carry the **same information** (text vs. number)
- The target has a **trailing period** in part of the data (`>50K.`)
- `fnlwgt` is a **census sampling weight**, not a personal feature


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score


## 0. Load the data
Use Option A (UCI) for reproducibility anywhere; Option B uses the uploaded CSVs.

In [ ]:
import numpy as np
import pandas as pd

!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo
adult = fetch_ucirepo(id=2)
X_raw, y_raw = adult.data.features.copy(), adult.data.targets.copy()
df = X_raw.copy(); df['income'] = y_raw.iloc[:, 0].values

In [ ]:
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [ ]:
df['income']

,income
0,<=50K
1,<=50K
2,<=50K
3,<=50K
4,<=50K
...,...
48837,<=50K.
48838,<=50K.
48839,<=50K.
48840,<=50K.


## 1. Standardize column names
Lowercase, and replace `.` and `-` with `_` (e.g. `education-num` → `education_num`).*italicized text*

In [ ]:

# Standardize column names
df.columns = [col.lower().replace('-', '_').replace('.', '_') for col in df.columns]
df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## 2. Strip whitespace in text columns
The raw UCI file stores values like `' Private'` with a leading space, so the same value can be treated as two. Strip it (safe to run even if already clean).

In [ ]:
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()
df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## 3. Turn the `?` marker into real missing values
Missing data hides as the string `?`, so `isna()` does not catch it. Convert it first so the missingness becomes visible.

In [ ]:
# Replace '?' with NaN (numpy is already imported as np)
df = df.replace('?', np.nan)
print(f"Missing values per column:\n{df.isna().sum()}")

Missing values per column:
age                  0
workclass         2799
fnlwgt               0
education            0
education_num        0
marital_status       0
occupation        2809
relationship         0
race                 0
sex                  0
capital_gain         0
capital_loss         0
hours_per_week       0
native_country     857
income               0
dtype: int64


## 4. Handle missing values — as an 'Unknown' category
Two valid options:
- **(a) Keep them as an 'Unknown' category** ← chosen here. The missingness may not be random (MNAR), so 'unknown' itself can be informative.
- (b) Impute the most frequent value (`SimpleImputer(strategy='most_frequent')`) — simpler, but erases information.

> **Think:** which fits this data better? In the homework, try both and compare how the distributions change.

In [ ]:
# Fill missing values with 'Unknown' for categorical columns
cat_cols = df.select_dtypes(include='object').columns
df[cat_cols] = df[cat_cols].fillna('Unknown')
df.isna().sum()

,0
age,0
workclass,0
fnlwgt,0
education,0
education_num,0
marital_status,0
occupation,0
relationship,0
race,0
sex,0


## 5. Drop duplicate rows
Exactly identical rows can distort training.

In [ ]:
# Drop duplicates
print(f"Rows before: {len(df)}")
df = df.drop_duplicates()
print(f"Rows after: {len(df)}")

Rows before: 48842
Rows after: 48813


## 6. Drop a redundant column
`education` (text) and `education_num` (number) carry the **same information**. The numeric one is already ordered, so we keep it and drop the text `education`.

> **Check:** confirm the 1-to-1 mapping yourself first.

In [ ]:
# Drop redundant education column
if 'education' in df.columns:
    df = df.drop(columns=['education'])
df.head()

,age,workclass,fnlwgt,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## 7. Clean the target → 0/1
Some rows (from the test split) have a trailing period like `>50K.`. Remove it, then map `>50K` to 1 and `<=50K` to 0.

In [ ]:
# Clean target: remove trailing periods and map to 0/1
df['income'] = df['income'].str.rstrip('.')
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})
df['income'].value_counts()

,count
income,
0,37128
1,11685


## 8. Drop `fnlwgt` (a research decision)
`fnlwgt` is a census **sampling weight**, not a feature describing the individual's income. We drop it for this study.

In [ ]:
# Drop census weight column
if 'fnlwgt' in df.columns:
    df = df.drop(columns=['fnlwgt'])
df.head()

,age,workclass,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,0
1,50,Self-emp-not-inc,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,0
2,38,Private,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,0
3,53,Private,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,0
4,28,Private,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,0


## 9. Split into X / y and final check

In [ ]:
# Final Split into features and target
X = df.drop('income', axis=1)
y = df['income']

print("Preprocessing Complete according to Hands-On ML standards.")
display(X.head())
display(y.head())

Preprocessing Complete according to Hands-On ML standards.


,age,workclass,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country
0,39,State-gov,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States
1,50,Self-emp-not-inc,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States
2,38,Private,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States
3,53,Private,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States
4,28,Private,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba


,income
0,0
1,0
2,0
3,0
4,0


In [ ]:
num = X.select_dtypes(exclude='object').columns.tolist()
cat = X.select_dtypes(include='object').columns.tolist()
print(num)
print(cat)

['age', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']
['workclass', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'native_country']


In [ ]:
from sklearn.model_selection import train_test_split

Xtr, Xte, ytr, yte =  train_test_split(X, y, test_size=0.2, random_state=42)


# Logistic Regression
1. 4-feature model
2. all feature model
3. all + engineered features model

## 1. 4-feature model

In [ ]:
num_4 = ["age", "education_num"]
cat_4 = ["occupation", "sex"]

pre_4 = ColumnTransformer([
    ("num", StandardScaler(), num_4),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_4)
])

In [ ]:
pipe_4 = Pipeline([
    ("pre", pre_4),
    ("clf", LogisticRegression(max_iter=2000))
])

In [ ]:
pipe_4.fit(Xtr, ytr)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'education_num']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['occupation', 'sex'])])),
                ('clf', LogisticRegression(max_iter=2000))])

In [ ]:
y_pred_4 = pipe_4.predict(Xte)
y_proba_4 = pipe_4.predict_proba(Xte)[:, 1]

In [ ]:
f1_4 = f1_score(yte, y_pred_4)
auc_4 = roc_auc_score(yte, y_proba_4)

print(f"4-Feature Model - F1 Score: {f1_4:.4f}")
print(f"4-Feature Model - AUC Score: {auc_4:.4f}")

4-Feature Model - F1 Score: 0.4987
4-Feature Model - AUC Score: 0.8243


## 2. all feature model


In [ ]:
Xtr[:3]

,age,workclass,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country
40819,70,Unknown,15,Divorced,Unknown,Not-in-family,White,Male,2538,0,45,United-States
40613,30,Private,8,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,44,United-States
45445,59,Private,13,Separated,Exec-managerial,Not-in-family,White,Male,0,0,45,United-States


In [ ]:
ytr[:3]

,income
40819,0
40613,0
45445,0


In [ ]:
pre_raw = ColumnTransformer([
    ('num', StandardScaler(), num ),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat)
])

In [ ]:

pipe_raw = Pipeline( [ ('pre', pre_raw),

            ( 'clf', LogisticRegression(max_iter=2000))

            ])

In [ ]:
pipe_raw.fit(Xtr, ytr)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'education_num',
                                                   'capital_gain',
                                                   'capital_loss',
                                                   'hours_per_week']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['workclass',
                                                   'marital_status',
                                                   'occupation', 'relationship',
                                                   'race', 'sex',
                                                   'native_country'])])),
                ('clf', LogisticRegression(max_iter=2000))])

## 3. all + engineered feature model


In [ ]:
def add_features(df_in):
    d = df_in.copy()
    d['capital_net'] = d['capital_gain'] - d['capital_loss']
    d['has_capital_gain'] = (d['capital_gain'] > 0).astype(int)
    d['log_capital_gain'] = np.log1p(d['capital_gain'])
    d['age_x_hours'] = d['age'] * d['hours_per_week']
    d['edu_x_hours'] = d['education_num'] * d['hours_per_week']
    d['age_sq'] = d['age'] ** 2
    return d

In [ ]:
Xtr_eng = add_features(Xtr)
Xte_eng = add_features(Xte)

num_eng = Xtr_eng.select_dtypes(exclude='object').columns.tolist()
cat_eng = Xtr_eng.select_dtypes(include='object').columns.tolist()

In [ ]:
pre_eng = ColumnTransformer([
    ('num', StandardScaler(), num_eng),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_eng)
])

pipe_eng = Pipeline([
    ('pre', pre_eng),
    ('clf', LogisticRegression(max_iter=2000))
])

In [ ]:
pipe_eng.fit(Xtr_eng, ytr)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'education_num',
                                                   'capital_gain',
                                                   'capital_loss',
                                                   'hours_per_week',
                                                   'capital_net',
                                                   'has_capital_gain',
                                                   'log_capital_gain',
                                                   'age_x_hours', 'edu_x_hours',
                                                   'age_sq']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['workclass',
                                                   'marital_status',
                                                   'occupation', 'relationship',
                                                   'race', 'sex',
                                                   'native_country'])])),
                ('clf', LogisticRegression(max_iter=2000))])

In [ ]:
y_pred_eng = pipe_eng.predict(Xte_eng)
y_proba_eng = pipe_eng.predict_proba(Xte_eng)[:, 1]

In [ ]:
f1_eng = f1_score(yte, y_pred_eng)
auc_eng = roc_auc_score(yte, y_proba_eng)

print(f"Engineered Model - F1 Score: {f1_eng:.4f}")
print(f"Engineered Model - AUC Score: {auc_eng:.4f}")

Engineered Model - F1 Score: 0.6768
Engineered Model - AUC Score: 0.9101


# Decision Tree
1. 4-feature model
2. all feature model
3. all + engineered features model

In [ ]:
display(Xtr.head())
display(ytr.head())

,age,workclass,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country
40819,70,Unknown,15,Divorced,Unknown,Not-in-family,White,Male,2538,0,45,United-States
40613,30,Private,8,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,44,United-States
45445,59,Private,13,Separated,Exec-managerial,Not-in-family,White,Male,0,0,45,United-States
31318,22,Private,10,Never-married,Adm-clerical,Unmarried,Black,Female,0,0,30,United-States
3719,21,Local-gov,10,Never-married,Adm-clerical,Own-child,White,Male,0,0,40,Guatemala


,income
40819,0
40613,0
45445,0
31318,0
3719,0


In [ ]:
from sklearn.tree import DecisionTreeClassifier

## 2. All features

In [ ]:
num

['age', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']

In [ ]:
cat

['workclass',
 'marital_status',
 'occupation',
 'relationship',
 'race',
 'sex',
 'native_country']

In [ ]:
pre_raw = ColumnTransformer(
    [
        ('num', StandardScaler(), num ),
        ('cat', OneHotEncoder(handle_unknown="ignore"), cat)

    ]

)

In [ ]:
pi = Pipeline(
    [
        ('pre', pre_raw ),
        ( 'clf', DecisionTreeClassifier(max_depth=8))

    ]
)

In [ ]:
pi.fit(Xtr, ytr)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'education_num',
                                                   'capital_gain',
                                                   'capital_loss',
                                                   'hours_per_week']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['workclass',
                                                   'marital_status',
                                                   'occupation', 'relationship',
                                                   'race', 'sex',
                                                   'native_country'])])),
                ('clf', DecisionTreeClassifier(max_depth=8))])

In [ ]:
phat = pi.predict_proba(Xte)
phat

array([[0.97817008, 0.02182992],
       [0.99693703, 0.00306297],
       [0.94015748, 0.05984252],
       ...,
       [0.        , 1.        ],
       [0.5316723 , 0.4683277 ],
       [0.99693703, 0.00306297]])

In [ ]:
roc_auc_score(yte, phat[:, 1])

np.float64(0.9023329358657695)


# Random Forest
1. 4-feature model
2. all feature model
3. all + engineered features model

# 4 feature

In [ ]:
num_4 = ['age', 'education_num']
cat_4 = ['occupation', 'sex']

In [ ]:
pre_4_rf = ColumnTransformer([
    ('num', StandardScaler(), num_4),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_4)
])

In [ ]:
pipe_4_rf = Pipeline([
    ('pre', pre_4_rf),
    ('clf', RandomForestClassifier())
])

In [ ]:
pipe_4_rf.fit(Xtr, ytr)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'education_num']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['occupation', 'sex'])])),
                ('clf', RandomForestClassifier())])

In [ ]:
y_pred_4_rf = pipe_4_rf.predict(Xte)
y_proba_4_rf = pipe_4_rf.predict_proba(Xte)[:, 1]

In [ ]:
f1_4_rf = f1_score(yte, y_pred_4_rf)
auc_4_rf = roc_auc_score(yte, y_proba_4_rf)

print(f"Random Forest 4-Features - F1 Score: {f1_4_rf:.4f}")
print(f"Random Forest 4-Features - AUC Score: {auc_4_rf:.4f}")

Random Forest 4-Features - F1 Score: 0.5128
Random Forest 4-Features - AUC Score: 0.7978


# All feature

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
pre_raw = ColumnTransformer(
    [
        ( 'num', StandardScaler(), num),
        ( 'cat', OneHotEncoder(handle_unknown="ignore"), cat),

    ]
)

In [ ]:
pipe = Pipeline(
    [
        ('pre', pre_raw ),
        ( 'clf', RandomForestClassifier() ),
    ]
)

In [ ]:
pipe.fit(Xtr, ytr)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'education_num',
                                                   'capital_gain',
                                                   'capital_loss',
                                                   'hours_per_week']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['workclass',
                                                   'marital_status',
                                                   'occupation', 'relationship',
                                                   'race', 'sex',
                                                   'native_country'])])),
                ('clf', RandomForestClassifier())])

In [ ]:
phat = pipe.predict_proba(Xte)[:, 1]
phat

array([0.01      , 0.00666667, 0.05      , ..., 1.        , 0.10792857,
       0.        ])

In [ ]:
roc_auc_score(yte, phat)

np.float64(0.8883771399369731)

## 3. All + engineered features model

In [ ]:
pre_eng_rf = ColumnTransformer([
    ("num", StandardScaler(), num_eng),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_eng)
])

In [ ]:
pipe_eng_rf = Pipeline([
    ("pre", pre_eng_rf),
    ("clf", RandomForestClassifier())
])

In [ ]:
pipe_eng_rf.fit(Xtr_eng, ytr)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'education_num',
                                                   'capital_gain',
                                                   'capital_loss',
                                                   'hours_per_week',
                                                   'capital_net',
                                                   'has_capital_gain',
                                                   'log_capital_gain',
                                                   'age_x_hours', 'edu_x_hours',
                                                   'age_sq']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['workclass',
                                                   'marital_status',
                                                   'occupation', 'relationship',
                                                   'race', 'sex',
                                                   'native_country'])])),
                ('clf', RandomForestClassifier())])

In [ ]:
y_pred_eng_rf = pipe_eng_rf.predict(Xte_eng)
y_proba_eng_rf = pipe_eng_rf.predict_proba(Xte_eng)[:, 1]

In [ ]:
f1_eng_rf = f1_score(yte, y_pred_eng_rf)
auc_eng_rf = roc_auc_score(yte, y_proba_eng_rf)

print(f"Random Forest Engineered - F1 Score: {f1_eng_rf:.4f}")
print(f"Random Forest Engineered - AUC Score: {auc_eng_rf:.4f}")

Random Forest Engineered - F1 Score: 0.6721
Random Forest Engineered - AUC Score: 0.8960
